# GAE-GeometricAutoEncoder: Kaggle Dual-GPU Pipeline

This notebook runs **Geometric AutoEncoder (GAE)** multi-view novel view synthesis on Kaggle's dual-T4 GPUs.
- **Stage 1**: Depth-Anything-3 (DA3-GIANT) 3D geometric feature extraction
- **Stage 2**: Flow Matching DiT generative novel view synthesis & 3D point cloud generation

In [ ]:
# Cell 1: Environment Setup, Repo Clone/Update & Dependency Installation
import os, sys, subprocess, shutil
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("TMPDIR", "/tmp")

ROOT = "/kaggle/working/GAE-GeometricAutoEncoder"
BRANCH = "feat/kaggle-runner"
REPO_URL = "https://github.com/A511-git/GAE-GeometricAutoEncoder"

def run_cmd(cmd):
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout[-1500:])
    if res.stderr and res.returncode != 0:
        print("[ERROR]", res.stderr[-800:])
    return res.returncode

# 1. Clone or pull repo
if not os.path.isdir(ROOT):
    print(f"--> Cloning {REPO_URL} (branch: {BRANCH})...")
    code = run_cmd(f"git clone -b {BRANCH} {REPO_URL} {ROOT}")
    if code != 0:
        print(f"--> Branch '{BRANCH}' not found on remote, cloning default branch...")
        run_cmd(f"git clone {REPO_URL} {ROOT}")
else:
    print("--> Repo already exists, pulling latest changes...")
    run_cmd(f"git -C {ROOT} fetch origin")
    run_cmd(f"git -C {ROOT} checkout {BRANCH} || git -C {ROOT} checkout main")
    run_cmd(f"git -C {ROOT} pull")

# 2. Configure sys.path
os.chdir(ROOT)
for p in (ROOT, f"{ROOT}/src", f"{ROOT}/scripts/eval"):
    if p not in sys.path:
        sys.path.insert(0, p)

# 3. Install compatible dependencies
print("--> Checking & installing dependencies...")
run_cmd("pip install -q 'numpy<2.3' 'scipy>=1.11' depth-anything-3 omegaconf einops timm torchdiffeq plyfile addict e3nn")
if os.path.isfile(f"{ROOT}/requirements.txt"):
    run_cmd(f"pip install -q -r {ROOT}/requirements.txt")

# 4. Sanity verification
import numpy, scipy, torch, cv2
import depth_anything_3
from gae.hub import download_weights, extract_da3_stats
print("\n[SUCCESS] Environment ready!")
print(f"- PyTorch: {torch.__version__} (CUDA available: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()})")
print(f"- NumPy: {numpy.__version__} | SciPy: {scipy.__version__} | OpenCV: {cv2.__version__}")

In [ ]:
# Cell 2: Download Model Checkpoints & Pre-extracted Normalization Stats
from gae.hub import download_weights, extract_da3_stats
from pathlib import Path

CKPT_DIR = f"{ROOT}/ckpts"
print("--> Downloading GAE weights from Hugging Face Hub (TencentARC/GAE-D64-1B)...")
paths = download_weights("TencentARC/GAE-D64-1B", size="64", out_dir=CKPT_DIR)

stats_dir = Path(ROOT) / "model_stats" / "da3_giant_5ds"
if "da3_stats_giant_5ds.tar" in paths:
    print("--> Extracting DA3 normalization stats...")
    extract_da3_stats(paths["da3_stats_giant_5ds.tar"], stats_dir)

if not (stats_dir / "normalization_stats_level0.pt").is_file():
    for f in stats_dir.rglob("normalization_stats_level*.pt"):
        shutil.move(str(f), str(stats_dir / f.name))

print("\n[Checkpoints Ready]")
for k, v in paths.items():
    print(f"  • {k}: {v}")
print("\n[Stats files]:", sorted(p.name for p in stats_dir.iterdir() if p.name.endswith('.pt')))

In [ ]:
# Cell 3: 2-GPU Model Parallelism Patch (Splits DA3-GIANT across cuda:0 and cuda:1)
import torch
from stage1.da3 import DA3Backbone
from stage1.encoders.da3 import DA3EncoderDirect

num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    DEV0, DEV1 = torch.device("cuda:0"), torch.device("cuda:1")
    SPLIT_AT = 20   # blocks [0, SPLIT_AT) on cuda:0, blocks [SPLIT_AT, ...) on cuda:1

    def _mv(t, dev):
        if t is None:
            return None
        if isinstance(t, (tuple, list)):
            return type(t)(_mv(x, dev) for x in t)
        return t.to(dev)

    def _run_backbone_loop_2gpu(self, x, B, S, H, W, target_layers=None, stop_at_first=False):
        trans = self.backbone.pretrained
        x = trans.prepare_tokens_with_masks(x)
        B_sq, S_tok, N, C_emb = x.shape
        pos_all, pos_nodiff_all = trans._prepare_rope(B, S, H, W, x.device)
        pos_all_1, pos_nodiff_all_1 = _mv(pos_all, DEV1), _mv(pos_nodiff_all, DEV1)

        current_x, local_x, results = x, None, {}
        if target_layers is None:
            target_layers = self.OUT_LAYERS

        for i, blk in enumerate(trans.blocks):
            if i == SPLIT_AT:
                current_x = current_x.to(DEV1)
                local_x = None if local_x is None else local_x.to(DEV1)
            on1 = i >= SPLIT_AT
            P_all, P_nodiff = (pos_all_1, pos_nodiff_all_1) if on1 else (pos_all, pos_nodiff_all)

            if i < trans.rope_start or trans.rope is None:
                g_pos, l_pos = None, None
            else:
                g_pos, l_pos = P_nodiff, P_all

            if trans.alt_start != -1 and i == trans.alt_start:
                ref_token = trans.camera_token[:, :1].expand(B, -1, -1)
                if S > 1:
                    src_token = trans.camera_token[:, 1:].expand(B, S - 1, -1)
                    cam_token = torch.cat([ref_token, src_token], dim=1)
                else:
                    cam_token = ref_token
                current_x[:, :, 0] = cam_token.to(current_x.device)

            if trans.alt_start != -1 and i >= trans.alt_start and i % 2 == 1:
                attn_type, pos_emb = "global", g_pos
            else:
                attn_type, pos_emb = "local", l_pos

            current_x = trans.process_attention(current_x, blk, attn_type=attn_type, pos=pos_emb)
            if attn_type == "local":
                local_x = current_x

            if i in target_layers:
                curr_sq = current_x.view(B * S, N, -1)
                loc_sq = local_x.view(B * S, N, -1)
                # Return intermediate features to cuda:0 for downstream heads
                results[i] = torch.cat([loc_sq, curr_sq], dim=-1).to(DEV0)
                if stop_at_first:
                    return results
        return results

    def _split_blocks(rae):
        enc = rae.encoder
        if getattr(enc, "_split_done", False):
            return
        for blk in enc.backbone.pretrained.blocks[SPLIT_AT:]:
            blk.to(DEV1)
        enc._split_done = True
        print(f"[2-GPU] DA3 blocks {SPLIT_AT}..{len(enc.backbone.pretrained.blocks)-1} placed on cuda:1")

    DA3EncoderDirect._run_backbone_loop = _run_backbone_loop_2gpu

    _orig_eval = DA3Backbone.eval
    def _eval_then_split(self, *a, **k):
        out = _orig_eval(self, *a, **k)
        _split_blocks(self)
        return out
    DA3Backbone.eval = _eval_then_split
    print(f"[SUCCESS] 2-GPU parallel patch installed (using {num_gpus} GPUs).")
else:
    print(f"[INFO] Detected {num_gpus} GPU(s). Running on standard single-GPU pipeline.")

In [ ]:
# Cell 4: Run Zero-Manual-Pose Multi-View Re-framing & Generative Inference
import argparse, cv2, numpy as np
from pathlib import Path
from scripts.demo import multiview_reframer as mr

# Configurable paths and parameters
IMG_DIR   = "/kaggle/input/datasets/newmailserver/test-img"  # Set to your image or dataset directory
ANCHOR    = "PXL_20260926_175340689.jpg"                   # Anchor filename or prefix
OUT       = Path("/kaggle/working/my_results")
H, W      = 378, 672                                       # Must be divisible by 14
PULL_BACK = 1.2                                            # Distance in meters to pull camera backward
FOV_SCALE = 0.85                                           # < 1.0 widens Field of View
TOTAL_VIEWS = 17                                           # Number of trajectory frames to generate
USE_ONLY_ANCHOR = True                                     # True: 1 anchor input -> 17 novel frames
PROMPT = "a photorealistic scene with sharp details and natural lighting"

# Fallback if custom dataset is not mounted yet: use repo example
if not Path(IMG_DIR).exists():
    print(f"[WARNING] {IMG_DIR} not found. Falling back to repo example image.")
    IMG_DIR = f"{ROOT}/examples/scenes/forest_lake_trail.jpg"
    ANCHOR = "forest_lake_trail.jpg"

all_paths = mr._collect_image_paths([Path(IMG_DIR)])
anchor_idx = mr._resolve_anchor_indices(all_paths, ANCHOR)[0]

def load_im(p):
    im = cv2.imread(str(p), cv2.IMREAD_COLOR)
    assert im is not None, f"Could not read image at {p}"
    return cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA)

if USE_ONLY_ANCHOR:
    image_paths = [all_paths[anchor_idx]]
    images = [load_im(all_paths[anchor_idx])]
    a_idx = 0
else:
    image_paths = all_paths
    images = [load_im(p) for p in all_paths]
    a_idx = anchor_idx

ns = argparse.Namespace(
    resolution=(H, W),
    total_views=TOTAL_VIEWS,
    pull_back=PULL_BACK,
    fov_scale=FOV_SCALE,
    prompt=PROMPT,
    fps=12,
)

OUT.mkdir(parents=True, exist_ok=True)
manifest, cond_num = mr._prepare_scene_for_anchor(ns, image_paths, images, a_idx, OUT)

argv = [
    "eval_generation.py",
    "--dit-ckpt", str(paths["flow_gae64.pt"]),
    "--vae-ckpt", str(paths["gae_64.pt"]),
    "--gld-config", f"{ROOT}/configs/gae_64.yaml",
    "--config", f"{ROOT}/configs/flow_gae64.yaml",
    "--scene-manifest", str(manifest),
    "--dataset", "scannetpp",
    "--mode", "generate",
    "--prompt", PROMPT,
    "--cond-num", str(cond_num),
    "--num-scenes", "1",
    "--num-views", str(TOTAL_VIEWS),
    "--total-views", str(TOTAL_VIEWS),
    "--resolution", str(H), str(W),
    "--sample-steps", "50",
    "--cfg-scale", "2.0",
    "--seed", "42",
    "--video-fps", "12",
    "--pc-stride", "4",
    "--output-dir", str(OUT),
    "--save-pointcloud",
]
sys.argv = argv

import eval_generation as eg
torch.cuda.empty_cache()
try:
    eg.main()
except SystemExit as e:
    print("Generation run complete (exit status:", e, ")")

print("\n--- GPU VRAM Consumption ---")
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} peak allocated:", round(torch.cuda.max_memory_allocated(i) / 1e9, 2), "GB")

In [ ]:
# Cell 5: Visualize Novel View & Inspect Output Artifacts
import matplotlib.pyplot as plt
from PIL import Image

sd = OUT / "scannetpp"
strip_path = sd / "000_rgb.png"
if strip_path.exists():
    strip = cv2.imread(str(strip_path))
    novel = strip[-H:, :]                       # Last frame is the novel pulled-back view
    novel_path = sd / "000_novel_view.png"
    cv2.imwrite(str(novel_path), novel)

print("\n[Generated Output Files in:", str(sd), "]")
for p in sorted(sd.iterdir()):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  • {p.name} ({size_mb:.2f} MB)")

# Display Input vs Novel View
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
axes[0].set_title("Input Conditioning / Anchor Image")
axes[0].axis("off")

if (sd / "000_novel_view.png").exists():
    axes[1].imshow(cv2.cvtColor(novel, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Generated Novel View (Pull-Back {PULL_BACK}m, FOV {FOV_SCALE})")
    axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Direct Bash CLI Shortcuts (Alternative Execution)
# Single-image 3D VAE reconstruction:
# !PYTHONPATH=/kaggle/working/GAE-GeometricAutoEncoder python scripts/demo/reconstruct_vae.py \
#     --image /kaggle/input/datasets/newmailserver/test-img/PXL_20260925_150409686.jpg \
#     --hf-repo TencentARC/GAE-D64-1B \
#     --pc-stride 1 \
#     --resolution 672 378 \
#     --cache-dir /kaggle/working/GAE-GeometricAutoEncoder/ckpts \
#     --output /kaggle/working/results/vae_recon

# Multi-view Re-framer:
# !PYTHONPATH=/kaggle/working/GAE-GeometricAutoEncoder python scripts/demo/multiview_reframer.py \
#     --images /kaggle/input/datasets/newmailserver/test-img \
#     --anchor PXL_20260926_175340689.jpg \
#     --pull-back 1.2 \
#     --fov-scale 0.85 \
#     --resolution 378 672 \
#     --output /kaggle/working/my_results/
